In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Tensor creation
x = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
y = torch.randn(2, 3)
z = torch.zeros(3, 3)

print("\nTensor x:", x)
print("Tensor y shape:", y.shape)
print("Tensor z:\n", z)

In [ ]:
# Autograd - Automatic Differentiation
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = 2 * x ** 2 + 3 * x
z = y.sum()

print("x:", x)
print("y:", y)
print("z:", z)

# Backward pass
z.backward()
print("\nGradients (dz/dx):", x.grad)

# Manual derivative check: dy/dx = 4x + 3
expected_grad = 4 * x.data + 3
print("Expected gradients:", expected_grad)

In [ ]:
# Define a Neural Network
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Create model
model = SimpleNet()
print("Model:\n", model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params}")

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=False, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Training function
def train_epoch(model, train_loader, criterion, optimizer):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

# Evaluation function
def evaluate(model, test_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
            
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    accuracy = correct / total
    return total_loss / len(test_loader), accuracy

# Setup
model = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Training setup complete!")

In [ ]:
# Convolutional Neural Network
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

cnn_model = ConvNet()
print("CNN Model:")
print(cnn_model)

# Count parameters
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"\nTotal parameters: {total_params}")

In [ ]:
# GPU/CPU device handling
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move model to device
model = ConvNet().to(device)
print("Model moved to device")

# Tensors also need to be moved
x = torch.randn(1, 1, 28, 28).to(device)
output = model(x)
print(f"Output shape: {output.shape}")
print(f"Output device: {output.device}")

## Key Concepts

### Autograd
- Dynamic computational graphs
- Automatic differentiation
- Eager execution

### Module System
- `nn.Module` base class
- Container layers (Sequential, ModuleList, ModuleDict)
- Parameter registration

### Training Loop
- Forward pass
- Loss computation
- Backward pass
- Parameter updates

### GPU Acceleration
- `.to(device)`
- `.cuda()` and `.cpu()`
- Device-agnostic code

### Performance Tips
- Use DataLoader for batching
- Pin memory for GPU transfer
- Use half precision (fp16)
- Profile bottlenecks

## Practice Exercises

1. **Autograd**: Build gradient computation
2. **Custom Models**: Implement custom architectures
3. **Training**: Complete training loops
4. **CNNs**: Image classification
5. **RNNs**: Sequence modeling